# Rugby League score-distribution research

## Historical matrix blended with:
1. Negative Binomial
2. Bivariate / correlated Poisson

This notebook is designed to sit **after the expected-score model**. It keeps the expected home and away scores fixed, changes only the shape of the score distribution, and evaluates every historical fixture using information available before that fixture.

### Required input columns

The analysis dataframe must contain:

- `fixture_id`
- `match_date`
- `season`
- `home_score`
- `away_score`
- `expected_home_score`
- `expected_away_score`

Optional columns such as teams can be retained.

### Research questions

- Does Negative Binomial overdispersion improve on independent Poisson?
- Does positive home-away score dependence improve on independent Poisson?
- Does either candidate improve the current tilted-historical blend?
- Are any improvements stable by season and across bootstrap samples?

The notebook deliberately exposes a small adapter cell for connecting it to the project's database or existing feature-building functions.


In [1]:
from __future__ import annotations

import math
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import gammaln, logsumexp
from scipy.stats import nbinom, poisson

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


## 1. Configuration

Update `DATABASE_PATH` and, if necessary, the SQL query. The default query assumes the realised scores are in `results` and historical expected scores are in `expected_scores`.

If your existing research notebook already creates the required dataframe, replace the SQL-loading cell with that dataframe and continue from the validation cell.


In [2]:
DATABASE_PATH = Path("../data/rugby_league_pricing.db")

MAX_SCORE = 80
MIN_HISTORY_MATCHES = 100

# Historical matrix recency settings.
FULL_WEIGHT_YEARS = 2
HALF_WEIGHT_YEARS = 4
OLDER_HALF_LIFE_YEARS = 3.0

# Blend weight = probability assigned to the tilted historical matrix.
HISTORICAL_WEIGHTS = np.round(np.arange(0.0, 1.0001, 0.05), 2)

# Negative Binomial alpha in Var(Y) = mu + alpha * mu^2.
NB_ALPHA_GRID = np.array([0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.12, 0.16, 0.24])

# Bivariate Poisson shared component:
# lambda_shared = shared_fraction * min(mu_home, mu_away).
BIVARIATE_SHARED_FRACTION_GRID = np.array(
    [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.12, 0.16, 0.20]
)

EPSILON = 1e-15


In [3]:
QUERY = '''
SELECT
    f.fixture_id,
    f.match_date,
    CAST(strftime('%Y', f.match_date) AS INTEGER) AS season,
    f.home_team_id,
    f.away_team_id,
    r.home_score,
    r.away_score,
    es.expected_home_score,
    es.expected_away_score
FROM fixtures AS f
JOIN results AS r
    ON r.fixture_id = f.fixture_id
JOIN expected_scores AS es
    ON es.fixture_id = f.fixture_id
ORDER BY f.match_date, f.fixture_id
'''

if DATABASE_PATH.exists():
    with sqlite3.connect(DATABASE_PATH) as connection:
        matches = pd.read_sql_query(QUERY, connection, parse_dates=["match_date"])
else:
    matches = pd.DataFrame()
    print(
        f"Database not found at {DATABASE_PATH.resolve()}. "
        "Set DATABASE_PATH or assign the required dataframe to `matches`."
    )


### Alternative adapter

A dataframe produced elsewhere can be normalised as follows:

```python
matches = existing_dataframe.rename(
    columns={
        "date": "match_date",
        "actual_home_score": "home_score",
        "actual_away_score": "away_score",
        "predicted_home_score": "expected_home_score",
        "predicted_away_score": "expected_away_score",
    }
)
```


In [4]:
REQUIRED_COLUMNS = {
    "fixture_id",
    "match_date",
    "season",
    "home_score",
    "away_score",
    "expected_home_score",
    "expected_away_score",
}

def validate_matches(frame: pd.DataFrame) -> pd.DataFrame:
    missing = REQUIRED_COLUMNS.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    validated = frame.copy()
    validated["match_date"] = pd.to_datetime(validated["match_date"])
    validated = validated.sort_values(["match_date", "fixture_id"]).reset_index(drop=True)

    numeric_columns = [
        "home_score",
        "away_score",
        "expected_home_score",
        "expected_away_score",
    ]
    validated[numeric_columns] = validated[numeric_columns].apply(pd.to_numeric)

    if validated[numeric_columns].isna().any().any():
        raise ValueError("Required score columns contain null values.")

    if (validated[["home_score", "away_score"]] < 0).any().any():
        raise ValueError("Actual scores cannot be negative.")

    if (validated[["expected_home_score", "expected_away_score"]] <= 0).any().any():
        raise ValueError("Expected scores must be positive.")

    if validated["fixture_id"].duplicated().any():
        raise ValueError("fixture_id must be unique in the analysis dataframe.")

    return validated


if not matches.empty:
    matches = validate_matches(matches)
    display(matches.head())
    print(f"{len(matches):,} completed fixtures loaded.")


,fixture_id,match_date,season,home_team_id,away_team_id,home_score,away_score,expected_home_score,expected_away_score
0,2010-01-30_3_4,2010-01-30,2010,3,4,10,18,6.000000,34.000000
1,2010-02-05_2_7,2010-02-05,2010,2,7,10,24,10.000000,21.794118
2,2010-02-05_5_6,2010-02-05,2010,5,6,24,12,8.000000,26.000000
3,2010-02-05_8_1,2010-02-05,2010,8,1,38,6,9.942029,21.760870
4,2010-02-06_9_10,2010-02-06,2010,9,10,12,32,17.600000,18.800000


3,103 completed fixtures loaded.


## 2. Core probability-matrix functions

All matrices use integer scores from `0` to `MAX_SCORE`.

The final row and column absorb any probability above `MAX_SCORE`, avoiding accidental loss of probability mass. For a production implementation, a higher cap can be used if desired.


In [5]:
def normalise_matrix(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=float)
    matrix = np.clip(matrix, 0.0, None)
    total = matrix.sum()
    if not np.isfinite(total) or total <= 0:
        raise ValueError("Probability matrix has invalid total mass.")
    return matrix / total


def independent_matrix(home_pmf: np.ndarray, away_pmf: np.ndarray) -> np.ndarray:
    return normalise_matrix(np.outer(home_pmf, away_pmf))


def poisson_pmf_with_tail(mean: float, max_score: int) -> np.ndarray:
    scores = np.arange(max_score + 1)
    pmf = poisson.pmf(scores, mean)
    pmf[-1] += max(0.0, 1.0 - pmf.sum())
    return pmf / pmf.sum()


def negative_binomial_pmf_with_tail(
    mean: float,
    alpha: float,
    max_score: int,
) -> np.ndarray:
    if alpha <= 1e-12:
        return poisson_pmf_with_tail(mean, max_score)

    # scipy nbinom: mean = n(1-p)/p, variance = n(1-p)/p^2.
    n = 1.0 / alpha
    p = n / (n + mean)
    scores = np.arange(max_score + 1)
    pmf = nbinom.pmf(scores, n, p)
    pmf[-1] += max(0.0, 1.0 - pmf.sum())
    return pmf / pmf.sum()


def independent_poisson_matrix(
    expected_home: float,
    expected_away: float,
    max_score: int = MAX_SCORE,
) -> np.ndarray:
    return independent_matrix(
        poisson_pmf_with_tail(expected_home, max_score),
        poisson_pmf_with_tail(expected_away, max_score),
    )


def independent_negative_binomial_matrix(
    expected_home: float,
    expected_away: float,
    alpha: float,
    max_score: int = MAX_SCORE,
) -> np.ndarray:
    return independent_matrix(
        negative_binomial_pmf_with_tail(expected_home, alpha, max_score),
        negative_binomial_pmf_with_tail(expected_away, alpha, max_score),
    )


In [6]:
def bivariate_poisson_matrix(
    expected_home: float,
    expected_away: float,
    shared_fraction: float,
    max_score: int = MAX_SCORE,
) -> np.ndarray:
    """Construct X=U+W and Y=V+W with independent Poisson components.

    E[X] and E[Y] remain exactly equal to the supplied expected scores.
    Cov(X, Y) = lambda_shared.
    """
    lambda_shared = shared_fraction * min(expected_home, expected_away)
    lambda_shared = min(lambda_shared, expected_home, expected_away)

    lambda_home_only = max(expected_home - lambda_shared, 0.0)
    lambda_away_only = max(expected_away - lambda_shared, 0.0)

    scores = np.arange(max_score + 1)
    log_matrix = np.full((max_score + 1, max_score + 1), -np.inf)

    for home_score in scores:
        for away_score in scores:
            max_shared = min(home_score, away_score)
            shared_scores = np.arange(max_shared + 1)

            terms = (
                poisson.logpmf(home_score - shared_scores, lambda_home_only)
                + poisson.logpmf(away_score - shared_scores, lambda_away_only)
                + poisson.logpmf(shared_scores, lambda_shared)
            )
            log_matrix[home_score, away_score] = logsumexp(terms)

    matrix = np.exp(log_matrix)

    # The final cells collectively absorb the small truncated tail.
    missing_mass = max(0.0, 1.0 - matrix.sum())
    matrix[-1, -1] += missing_mass
    return normalise_matrix(matrix)


## 3. Leakage-safe historical matrix and exponential tilting

For fixture date `t`, the historical prior contains only matches strictly before `t`.

The empirical matrix is then exponentially tilted:

\[
P^*(h,a) \propto P_0(h,a)\exp(\theta_h h + \theta_a a)
\]

The two tilt parameters are solved so that the resulting matrix has the required expected home and away scores.


In [7]:
def historical_match_weights(
    prior_matches: pd.DataFrame,
    as_of_date: pd.Timestamp,
) -> np.ndarray:
    age_years = (
        (as_of_date - prior_matches["match_date"]).dt.days.to_numpy(dtype=float)
        / 365.25
    )

    weights = np.ones(len(prior_matches), dtype=float)

    middle = (age_years > FULL_WEIGHT_YEARS) & (age_years <= HALF_WEIGHT_YEARS)
    weights[middle] = 0.5

    older = age_years > HALF_WEIGHT_YEARS
    extra_age = age_years[older] - HALF_WEIGHT_YEARS
    weights[older] = 0.5 * np.power(0.5, extra_age / OLDER_HALF_LIFE_YEARS)

    return weights


def build_weighted_historical_matrix(
    prior_matches: pd.DataFrame,
    as_of_date: pd.Timestamp,
    max_score: int = MAX_SCORE,
    smoothing: float = 1e-6,
) -> np.ndarray:
    weights = historical_match_weights(prior_matches, as_of_date)
    matrix = np.full((max_score + 1, max_score + 1), smoothing, dtype=float)

    home_scores = np.minimum(
        prior_matches["home_score"].to_numpy(dtype=int),
        max_score,
    )
    away_scores = np.minimum(
        prior_matches["away_score"].to_numpy(dtype=int),
        max_score,
    )

    np.add.at(matrix, (home_scores, away_scores), weights)
    return normalise_matrix(matrix)


def matrix_moments(matrix: np.ndarray) -> tuple[float, float, np.ndarray]:
    scores = np.arange(matrix.shape[0], dtype=float)
    home_mean = float((matrix * scores[:, None]).sum())
    away_mean = float((matrix * scores[None, :]).sum())

    centred_home = scores[:, None] - home_mean
    centred_away = scores[None, :] - away_mean

    covariance = np.array(
        [
            [
                (matrix * centred_home * centred_home).sum(),
                (matrix * centred_home * centred_away).sum(),
            ],
            [
                (matrix * centred_home * centred_away).sum(),
                (matrix * centred_away * centred_away).sum(),
            ],
        ],
        dtype=float,
    )
    return home_mean, away_mean, covariance


def tilt_matrix_to_means(
    base_matrix: np.ndarray,
    target_home: float,
    target_away: float,
    tolerance: float = 1e-9,
    max_iterations: int = 100,
) -> np.ndarray:
    base = normalise_matrix(base_matrix)
    scores = np.arange(base.shape[0], dtype=float)
    home_grid = scores[:, None]
    away_grid = scores[None, :]

    theta = np.zeros(2, dtype=float)

    for _ in range(max_iterations):
        log_weights = (
            np.log(np.clip(base, EPSILON, None))
            + theta[0] * home_grid
            + theta[1] * away_grid
        )
        log_weights -= logsumexp(log_weights)
        tilted = np.exp(log_weights)

        home_mean, away_mean, covariance = matrix_moments(tilted)
        error = np.array(
            [home_mean - target_home, away_mean - target_away],
            dtype=float,
        )

        if np.max(np.abs(error)) < tolerance:
            return normalise_matrix(tilted)

        covariance += np.eye(2) * 1e-10
        step = np.linalg.solve(covariance, error)

        # Damp large Newton steps for numerical stability.
        step_scale = max(1.0, np.max(np.abs(step)) / 1.5)
        theta -= step / step_scale

    raise RuntimeError(
        "Exponential tilt did not converge. "
        f"Targets were ({target_home:.3f}, {target_away:.3f})."
    )


def blend_matrices(
    historical_matrix: np.ndarray,
    theoretical_matrix: np.ndarray,
    historical_weight: float,
) -> np.ndarray:
    return normalise_matrix(
        historical_weight * historical_matrix
        + (1.0 - historical_weight) * theoretical_matrix
    )


## 4. Diagnostics for dispersion and score dependence

These do not select the final model, but they reveal whether the candidates address visible residual structure.

- Negative Binomial is supported when realised score variance exceeds the level implied by the expected means.
- Bivariate Poisson is supported when home and away residual scores remain positively associated after expected scores are removed.


In [8]:
def distribution_diagnostics(frame: pd.DataFrame) -> pd.Series:
    home_residual = frame["home_score"] - frame["expected_home_score"]
    away_residual = frame["away_score"] - frame["expected_away_score"]

    home_alpha_moment = (
        ((home_residual ** 2 - frame["expected_home_score"]).sum())
        / np.square(frame["expected_home_score"]).sum()
    )
    away_alpha_moment = (
        ((away_residual ** 2 - frame["expected_away_score"]).sum())
        / np.square(frame["expected_away_score"]).sum()
    )

    residual_covariance = np.cov(home_residual, away_residual, ddof=1)[0, 1]
    residual_correlation = np.corrcoef(home_residual, away_residual)[0, 1]

    return pd.Series(
        {
            "home_score_mean": frame["home_score"].mean(),
            "home_score_variance": frame["home_score"].var(ddof=1),
            "away_score_mean": frame["away_score"].mean(),
            "away_score_variance": frame["away_score"].var(ddof=1),
            "home_alpha_moment_estimate": max(home_alpha_moment, 0.0),
            "away_alpha_moment_estimate": max(away_alpha_moment, 0.0),
            "home_away_residual_covariance": residual_covariance,
            "home_away_residual_correlation": residual_correlation,
        }
    )


if not matches.empty:
    display(distribution_diagnostics(matches).to_frame("value"))


,value
home_score_mean,24.688688
home_score_variance,194.087451
away_score_mean,20.897519
away_score_variance,160.631016
home_alpha_moment_estimate,0.211297
away_alpha_moment_estimate,0.221216
home_away_residual_covariance,-43.867435
home_away_residual_correlation,-0.277114


## 5. Pricing metrics

The notebook evaluates:

- exact-score log loss;
- match-result log loss;
- match-result Brier score;
- handicap log loss at a supplied or model-derived line;
- totals log loss at a supplied or model-derived line.

For clean comparison across candidate distributions, the default handicap line is the expected margin and the default totals line is the expected total, both rounded to the nearest half-point.


In [9]:
def nearest_half(value: float) -> float:
    return round(value * 2.0) / 2.0


def outcome_probabilities(matrix: np.ndarray) -> np.ndarray:
    home_win = np.tril(matrix, k=-1).sum()
    draw = np.trace(matrix)
    away_win = np.triu(matrix, k=1).sum()
    return np.array([home_win, draw, away_win], dtype=float)


def actual_result_index(home_score: int, away_score: int) -> int:
    if home_score > away_score:
        return 0
    if home_score == away_score:
        return 1
    return 2


def probability_home_covers(matrix: np.ndarray, home_handicap: float) -> float:
    scores = np.arange(matrix.shape[0])
    adjusted_margin = scores[:, None] - scores[None, :] + home_handicap
    return float(matrix[adjusted_margin > 0].sum())


def probability_over(matrix: np.ndarray, total_line: float) -> float:
    scores = np.arange(matrix.shape[0])
    total_grid = scores[:, None] + scores[None, :]
    return float(matrix[total_grid > total_line].sum())


def score_matrix_metrics(
    matrix: np.ndarray,
    actual_home: int,
    actual_away: int,
    expected_home: float,
    expected_away: float,
    home_handicap: float | None = None,
    total_line: float | None = None,
) -> dict[str, float]:
    capped_home = min(int(actual_home), matrix.shape[0] - 1)
    capped_away = min(int(actual_away), matrix.shape[1] - 1)

    exact_probability = float(matrix[capped_home, capped_away])
    exact_score_log_loss = -math.log(max(exact_probability, EPSILON))

    result_probabilities = outcome_probabilities(matrix)
    result_index = actual_result_index(actual_home, actual_away)
    result_probability = result_probabilities[result_index]
    match_result_log_loss = -math.log(max(result_probability, EPSILON))

    actual_result = np.zeros(3, dtype=float)
    actual_result[result_index] = 1.0
    match_result_brier = float(np.square(result_probabilities - actual_result).sum())

    if home_handicap is None:
        home_handicap = -nearest_half(expected_home - expected_away)
        if float(home_handicap).is_integer():
            home_handicap -= 0.5

    home_cover_probability = probability_home_covers(matrix, home_handicap)
    actual_home_cover = float(actual_home - actual_away + home_handicap > 0)
    handicap_probability = (
        home_cover_probability if actual_home_cover else 1.0 - home_cover_probability
    )
    handicap_log_loss = -math.log(max(handicap_probability, EPSILON))

    if total_line is None:
        total_line = nearest_half(expected_home + expected_away)
        if float(total_line).is_integer():
            total_line += 0.5

    over_probability = probability_over(matrix, total_line)
    actual_over = float(actual_home + actual_away > total_line)
    total_probability = over_probability if actual_over else 1.0 - over_probability
    totals_log_loss = -math.log(max(total_probability, EPSILON))

    return {
        "exact_score_log_loss": exact_score_log_loss,
        "match_result_log_loss": match_result_log_loss,
        "match_result_brier": match_result_brier,
        "handicap_log_loss": handicap_log_loss,
        "totals_log_loss": totals_log_loss,
    }


## 6. Walk-forward experiment

For each evaluation fixture:

1. take only earlier fixtures;
2. build and tilt the historical matrix;
3. construct each theoretical candidate;
4. blend across the historical-weight grid;
5. save fixture-level losses.

The result table is intentionally long-form, making it easy to aggregate by model, parameter, blend weight, season, or fixture.


In [32]:
import json


@dataclass(frozen=True)
class Candidate:
    family: str
    parameter_name: str
    parameter_value: float


def candidate_grid() -> list[Candidate]:
    candidates = [
        Candidate("poisson", "none", 0.0),
    ]
    candidates.extend(
        Candidate("negative_binomial", "alpha", float(alpha))
        for alpha in NB_ALPHA_GRID
    )
    candidates.extend(
        Candidate("bivariate_poisson", "shared_fraction", float(value))
        for value in BIVARIATE_SHARED_FRACTION_GRID
    )
    return candidates


def theoretical_matrix(
    candidate: Candidate,
    expected_home: float,
    expected_away: float,
    max_score: int,
) -> np.ndarray:
    if candidate.family == "poisson":
        return independent_poisson_matrix(
            expected_home,
            expected_away,
            max_score,
        )

    if candidate.family == "negative_binomial":
        return independent_negative_binomial_matrix(
            expected_home,
            expected_away,
            alpha=candidate.parameter_value,
            max_score=max_score,
        )

    if candidate.family == "bivariate_poisson":
        return bivariate_poisson_matrix(
            expected_home,
            expected_away,
            shared_fraction=candidate.parameter_value,
            max_score=max_score,
        )

    raise ValueError(
        f"Unknown candidate family: {candidate.family}"
    )


def run_checkpointed_walk_forward_backtest(
    frame: pd.DataFrame,
    checkpoint_path: str | Path,
    *,
    min_history_matches: int = MIN_HISTORY_MATCHES,
    max_score: int = MAX_SCORE,
    historical_weights: Iterable[float] = HISTORICAL_WEIGHTS,
    candidates: Iterable[Candidate] | None = None,
    fixture_limit: int | None = None,
) -> pd.DataFrame:
    """Run a resumable walk-forward backtest.

    Each completed fixture is written to SQLite immediately.

    When rerun, fixture IDs already present in the checkpoint database are
    skipped automatically. If fixture_limit is supplied, the function
    processes that many new fixtures, rather than simply examining that many
    fixture rows.

    Parameters
    ----------
    frame:
        Historical match dataframe.

    checkpoint_path:
        SQLite file used to store completed fixture results.

    min_history_matches:
        Minimum number of prior matches required before a fixture can be
        evaluated.

    max_score:
        Maximum score represented by each probability matrix.

    historical_weights:
        Blend weights assigned to the tilted historical matrix.

    candidates:
        The theoretical distribution candidates to evaluate.

    fixture_limit:
        Maximum number of unfinished fixtures to process in this run.
        Leave as None to process every remaining fixture.
    """
    frame = validate_matches(frame)
    candidates = list(candidates or candidate_grid())
    historical_weights = [
        float(weight)
        for weight in historical_weights
    ]

    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with sqlite3.connect(checkpoint_path) as connection:
        connection.execute(
            """
            CREATE TABLE IF NOT EXISTS backtest_results (
                fixture_id TEXT NOT NULL,
                match_date TEXT NOT NULL,
                season INTEGER NOT NULL,
                family TEXT NOT NULL,
                parameter_name TEXT NOT NULL,
                parameter_value REAL NOT NULL,
                historical_weight REAL NOT NULL,
                exact_score_log_loss REAL NOT NULL,
                match_result_log_loss REAL NOT NULL,
                match_result_brier REAL NOT NULL,
                handicap_log_loss REAL NOT NULL,
                totals_log_loss REAL NOT NULL,
                PRIMARY KEY (
                    fixture_id,
                    family,
                    parameter_value,
                    historical_weight
                )
            )
            """
        )

        completed_fixture_ids = {
            str(row[0])
            for row in connection.execute(
                """
                SELECT DISTINCT fixture_id
                FROM backtest_results
                """
            ).fetchall()
        }

    eligible_indices: list[int] = []

    for frame_index, fixture in frame.iterrows():
        prior_count = int(
            (
                frame["match_date"]
                < fixture["match_date"]
            ).sum()
        )

        if prior_count >= min_history_matches:
            eligible_indices.append(frame_index)

    total_eligible = len(eligible_indices)

    fixtures_processed_this_run = 0
    fixtures_skipped = 0

    for position, frame_index in enumerate(
        eligible_indices,
        start=1,
    ):
        fixture = frame.loc[frame_index]
        fixture_id = str(fixture["fixture_id"])

        if fixture_id in completed_fixture_ids:
            fixtures_skipped += 1
            continue

        prior = frame.loc[
            frame["match_date"]
            < fixture["match_date"]
        ]

        expected_home = float(
            fixture["expected_home_score"]
        )
        expected_away = float(
            fixture["expected_away_score"]
        )

        fixture_max_score = max(
            max_score,
            int(np.ceil(expected_home + 25)),
            int(np.ceil(expected_away + 25)),
        )

        if fixture_max_score > max_score:
            print(
                f"Expanding matrix for fixture_id={fixture_id}: "
                f"{max_score} -> {fixture_max_score} "
                f"| expected=({expected_home:.3f}, {expected_away:.3f})"
            )

        historical_base = build_weighted_historical_matrix(
            prior,
            as_of_date=fixture["match_date"],
            max_score=fixture_max_score,
        )

        historical_tilted = tilt_matrix_to_means(
            historical_base,
            target_home=expected_home,
            target_away=expected_away,
        )

        fixture_records: list[
            dict[str, float | int | str]
        ] = []

        for candidate in candidates:
            candidate_matrix = theoretical_matrix(
                candidate,
                expected_home,
                expected_away,
                fixture_max_score,
            )

            for historical_weight in historical_weights:
                blended = blend_matrices(
                    historical_tilted,
                    candidate_matrix,
                    historical_weight=historical_weight,
                )

                metrics = score_matrix_metrics(
                    blended,
                    actual_home=int(
                        fixture["home_score"]
                    ),
                    actual_away=int(
                        fixture["away_score"]
                    ),
                    expected_home=expected_home,
                    expected_away=expected_away,
                )

                fixture_records.append(
                    {
                        "fixture_id": fixture_id,
                        "match_date": pd.Timestamp(
                            fixture["match_date"]
                        ).isoformat(),
                        "season": int(
                            fixture["season"]
                        ),
                        "family": candidate.family,
                        "parameter_name": (
                            candidate.parameter_name
                        ),
                        "parameter_value": (
                            candidate.parameter_value
                        ),
                        "historical_weight": (
                            historical_weight
                        ),
                        **metrics,
                    }
                )

        fixture_results = pd.DataFrame.from_records(
            fixture_records
        )

        # Save this fixture immediately.
        with sqlite3.connect(
            checkpoint_path
        ) as connection:
            fixture_results.to_sql(
                "backtest_results_staging",
                connection,
                if_exists="replace",
                index=False,
            )

            connection.execute(
                """
                INSERT OR REPLACE INTO backtest_results
                SELECT *
                FROM backtest_results_staging
                """
            )

            connection.execute(
                """
                DROP TABLE backtest_results_staging
                """
            )

            connection.commit()

        completed_fixture_ids.add(fixture_id)
        fixtures_processed_this_run += 1

        print(
            f"[{position:,}/{total_eligible:,}] "
            f"fixture_id={fixture_id} "
            f"| new={fixtures_processed_this_run:,} "
            f"| skipped={fixtures_skipped:,}"
        )

        if (
            fixture_limit is not None
            and fixtures_processed_this_run
            >= fixture_limit
        ):
            print(
                f"Reached fixture_limit="
                f"{fixture_limit:,}. "
                "Stopping cleanly."
            )
            break

    with sqlite3.connect(
        checkpoint_path
    ) as connection:
        results = pd.read_sql_query(
            """
            SELECT *
            FROM backtest_results
            ORDER BY
                match_date,
                fixture_id,
                family,
                parameter_value,
                historical_weight
            """,
            connection,
            parse_dates=["match_date"],
        )

    completed_count = (
        results["fixture_id"].nunique()
        if not results.empty
        else 0
    )

    remaining_count = max(
        total_eligible - completed_count,
        0,
    )

    print(
        f"Run complete. Processed "
        f"{fixtures_processed_this_run:,} new fixtures. "
        f"Checkpoint contains "
        f"{completed_count:,} fixtures. "
        f"Approximately "
        f"{remaining_count:,} eligible fixtures remain."
    )

    return results

In [33]:
nb_candidates

[Candidate(family='negative_binomial', parameter_name='alpha', parameter_value=0.0),
 Candidate(family='negative_binomial', parameter_name='alpha', parameter_value=0.01),
 Candidate(family='negative_binomial', parameter_name='alpha', parameter_value=0.02),
 Candidate(family='negative_binomial', parameter_name='alpha', parameter_value=0.04),
 Candidate(family='negative_binomial', parameter_name='alpha', parameter_value=0.06),
 Candidate(family='negative_binomial', parameter_name='alpha', parameter_value=0.08),
 Candidate(family='negative_binomial', parameter_name='alpha', parameter_value=0.12),
 Candidate(family='negative_binomial', parameter_name='alpha', parameter_value=0.16),
 Candidate(family='negative_binomial', parameter_name='alpha', parameter_value=0.24)]

In [34]:
nb_results = run_checkpointed_walk_forward_backtest(
    matches,
    checkpoint_path=(
        "../data/negative_binomial_backtest.sqlite"
    ),
    candidates=nb_candidates,
)

Expanding matrix for fixture_id=2011-08-14_14_4: 80 -> 89 | expected=(63.214, 13.161)
[287/3,003] fixture_id=2011-08-14_14_4 | new=1 | skipped=286
[288/3,003] fixture_id=2011-08-14_6_11 | new=2 | skipped=286
[289/3,003] fixture_id=2011-08-14_8_12 | new=3 | skipped=286
[290/3,003] fixture_id=2011-08-19_12_5 | new=4 | skipped=286
[291/3,003] fixture_id=2011-08-19_8_6 | new=5 | skipped=286
[292/3,003] fixture_id=2011-08-20_13_14 | new=6 | skipped=286
[293/3,003] fixture_id=2011-08-20_3_2 | new=7 | skipped=286
[294/3,003] fixture_id=2011-08-20_4_7 | new=8 | skipped=286
[295/3,003] fixture_id=2011-08-21_11_9 | new=9 | skipped=286
[296/3,003] fixture_id=2011-08-21_1_10 | new=10 | skipped=286
[297/3,003] fixture_id=2011-08-27_2_8 | new=11 | skipped=286
[298/3,003] fixture_id=2011-09-02_2_4 | new=12 | skipped=286
[299/3,003] fixture_id=2011-09-02_9_12 | new=13 | skipped=286
[300/3,003] fixture_id=2011-09-03_13_11 | new=14 | skipped=286
[301/3,003] fixture_id=2011-09-03_7_10 | new=15 | skipped=

RuntimeError: Exponential tilt did not converge. Targets were (20.791, 50.542).

In [ ]:
# This is the main compute-heavy cell.
#
# For a fast smoke test first, use:
# research_sample = matches.tail(300)
# backtest_results = run_walk_forward_backtest(
#     research_sample,
#     min_history_matches=100,
#     max_score=60,
# )
#
# For the full run:
#
# backtest_results = run_walk_forward_backtest(matches)
# backtest_results.to_parquet(
#     "../data/distribution_candidate_backtest.parquet",
#     index=False,
# )


In [ ]:
nb_candidates = [
    Candidate(
        family="negative_binomial",
        parameter_name="alpha",
        parameter_value=float(alpha),
    )
    for alpha in NB_ALPHA_GRID
]

In [27]:
nb_results = run_checkpointed_walk_forward_backtest(
    matches,
    checkpoint_path="../data/negative_binomial_backtest.sqlite",
    candidates=nb_candidates,
    max_score=60,
)

[201/3,003] fixture_id=2011-05-13_12_10 | new=1 | skipped=200
[202/3,003] fixture_id=2011-05-13_14_7 | new=2 | skipped=200
[203/3,003] fixture_id=2011-05-13_1_4 | new=3 | skipped=200
[204/3,003] fixture_id=2011-05-13_2_13 | new=4 | skipped=200
[205/3,003] fixture_id=2011-05-13_8_3 | new=5 | skipped=200
[206/3,003] fixture_id=2011-05-14_5_9 | new=6 | skipped=200
[207/3,003] fixture_id=2011-05-15_11_6 | new=7 | skipped=200
[208/3,003] fixture_id=2011-05-20_12_11 | new=8 | skipped=200
[209/3,003] fixture_id=2011-05-20_2_3 | new=9 | skipped=200
[210/3,003] fixture_id=2011-05-21_4_7 | new=10 | skipped=200
[211/3,003] fixture_id=2011-05-22_5_13 | new=11 | skipped=200
[212/3,003] fixture_id=2011-05-22_6_8 | new=12 | skipped=200
[213/3,003] fixture_id=2011-05-27_2_14 | new=13 | skipped=200
[214/3,003] fixture_id=2011-05-27_9_1 | new=14 | skipped=200
[215/3,003] fixture_id=2011-05-29_10_5 | new=15 | skipped=200
[216/3,003] fixture_id=2011-05-29_4_13 | new=16 | skipped=200
[217/3,003] fixture_id

RuntimeError: Exponential tilt did not converge. Targets were (63.214, 13.161).

In [ ]:
# This is the main compute-heavy cell.
#
# For a fast smoke test first, use:
# research_sample = matches.tail(300)
# backtest_results = run_walk_forward_backtest(
#     research_sample,
#     min_history_matches=100,
#     max_score=60,
# )
#
# For the full run:
#
# backtest_results = run_walk_forward_backtest(matches)
# backtest_results.to_parquet(
#     "../data/distribution_candidate_backtest.parquet",
#     index=False,
# )


## 7. Aggregate and compare candidates

Lower values are better for every reported metric.

The first table finds each family's best parameter and historical blend weight by exact-score log loss. The second exposes the full trade-off across metrics.


In [ ]:
METRIC_COLUMNS = [
    "exact_score_log_loss",
    "match_result_log_loss",
    "match_result_brier",
    "handicap_log_loss",
    "totals_log_loss",
]


def aggregate_results(results: pd.DataFrame) -> pd.DataFrame:
    return (
        results.groupby(
            [
                "family",
                "parameter_name",
                "parameter_value",
                "historical_weight",
            ],
            as_index=False,
        )[METRIC_COLUMNS]
        .mean()
        .sort_values("exact_score_log_loss")
        .reset_index(drop=True)
    )


def best_configuration_per_family(results: pd.DataFrame) -> pd.DataFrame:
    aggregate = aggregate_results(results)
    best_indices = aggregate.groupby("family")["exact_score_log_loss"].idxmin()
    return (
        aggregate.loc[best_indices]
        .sort_values("exact_score_log_loss")
        .reset_index(drop=True)
    )


# Example:
# aggregate = aggregate_results(backtest_results)
# best_overall = best_configuration_per_family(backtest_results)
# display(best_overall)
# display(aggregate.head(30))


In [ ]:
def plot_blend_curves(
    results: pd.DataFrame,
    metric: str = "exact_score_log_loss",
) -> None:
    aggregate = aggregate_results(results)

    best_parameter_rows = (
        aggregate.groupby(["family", "parameter_value"], as_index=False)[metric]
        .mean()
        .sort_values(metric)
        .groupby("family", as_index=False)
        .first()
    )

    plt.figure(figsize=(10, 6))

    for _, best_parameter in best_parameter_rows.iterrows():
        family = best_parameter["family"]
        parameter_value = best_parameter["parameter_value"]

        subset = aggregate[
            (aggregate["family"] == family)
            & (aggregate["parameter_value"] == parameter_value)
        ].sort_values("historical_weight")

        label = f"{family}: parameter={parameter_value:g}"
        plt.plot(
            subset["historical_weight"],
            subset[metric],
            marker="o",
            label=label,
        )

    plt.xlabel("Historical matrix weight")
    plt.ylabel(metric.replace("_", " ").title())
    plt.title(f"{metric.replace('_', ' ').title()} by blend weight")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


# Example:
# plot_blend_curves(backtest_results, "exact_score_log_loss")
# plot_blend_curves(backtest_results, "match_result_log_loss")
# plot_blend_curves(backtest_results, "totals_log_loss")


## 8. Season-by-season stability

A production candidate should not win only because of one season. This section reports the best in-sample configuration within each season and also allows a fixed configuration to be assessed across every season.


In [ ]:
def best_by_season(results: pd.DataFrame) -> pd.DataFrame:
    season_aggregate = (
        results.groupby(
            [
                "season",
                "family",
                "parameter_name",
                "parameter_value",
                "historical_weight",
            ],
            as_index=False,
        )[METRIC_COLUMNS]
        .mean()
    )

    best_indices = season_aggregate.groupby(
        ["season", "family"]
    )["exact_score_log_loss"].idxmin()

    return (
        season_aggregate.loc[best_indices]
        .sort_values(["season", "exact_score_log_loss"])
        .reset_index(drop=True)
    )


def evaluate_fixed_configuration_by_season(
    results: pd.DataFrame,
    family: str,
    parameter_value: float,
    historical_weight: float,
) -> pd.DataFrame:
    selected = results[
        (results["family"] == family)
        & np.isclose(results["parameter_value"], parameter_value)
        & np.isclose(results["historical_weight"], historical_weight)
    ]

    return (
        selected.groupby("season", as_index=False)[METRIC_COLUMNS]
        .agg(["mean", "count"])
    )


# Example:
# display(best_by_season(backtest_results))


## 9. Paired bootstrap against the current benchmark

The bootstrap resamples fixtures, not individual candidate rows. This preserves paired comparison: every candidate is assessed on exactly the same resampled games.

Set the benchmark to the current production candidate, for example:

- family: `poisson`
- parameter: `0.0`
- historical weight: `0.90` or `0.95`


In [ ]:
def select_configuration(
    results: pd.DataFrame,
    family: str,
    parameter_value: float,
    historical_weight: float,
) -> pd.DataFrame:
    selected = results[
        (results["family"] == family)
        & np.isclose(results["parameter_value"], parameter_value)
        & np.isclose(results["historical_weight"], historical_weight)
    ].copy()

    if selected.empty:
        raise ValueError("Requested configuration is not present in results.")

    return selected


def paired_bootstrap_difference(
    results: pd.DataFrame,
    metric: str,
    benchmark: tuple[str, float, float],
    candidate: tuple[str, float, float],
    repetitions: int = 2_000,
    seed: int = RANDOM_SEED,
) -> pd.Series:
    benchmark_frame = select_configuration(results, *benchmark)[
        ["fixture_id", metric]
    ].rename(columns={metric: "benchmark"})

    candidate_frame = select_configuration(results, *candidate)[
        ["fixture_id", metric]
    ].rename(columns={metric: "candidate"})

    paired = benchmark_frame.merge(
        candidate_frame,
        on="fixture_id",
        how="inner",
        validate="one_to_one",
    )
    paired["difference"] = paired["candidate"] - paired["benchmark"]

    random = np.random.default_rng(seed)
    differences = paired["difference"].to_numpy()
    sample_size = len(differences)

    bootstrap_means = np.empty(repetitions, dtype=float)
    for index in range(repetitions):
        sampled_indices = random.integers(0, sample_size, size=sample_size)
        bootstrap_means[index] = differences[sampled_indices].mean()

    lower, upper = np.quantile(bootstrap_means, [0.025, 0.975])

    return pd.Series(
        {
            "fixtures": sample_size,
            "mean_candidate_minus_benchmark": differences.mean(),
            "bootstrap_95pct_lower": lower,
            "bootstrap_95pct_upper": upper,
            "probability_candidate_better": float(
                np.mean(bootstrap_means < 0)
            ),
        }
    )


# Example:
# benchmark = ("poisson", 0.0, 0.90)
# nb_candidate = ("negative_binomial", 0.06, 0.90)
# biv_candidate = ("bivariate_poisson", 0.06, 0.90)
#
# display(
#     paired_bootstrap_difference(
#         backtest_results,
#         metric="exact_score_log_loss",
#         benchmark=benchmark,
#         candidate=nb_candidate,
#     )
# )


## 10. Optional nested walk-forward parameter selection

The main grid tells us which settings performed best over the full evaluation period. For a stricter estimate of deployable performance, use nested selection:

- before each season, select `alpha`, shared fraction, and blend weight using earlier seasons only;
- apply that fixed configuration to the next season;
- concatenate the held-out season results.

This prevents the evaluation season from influencing its own hyperparameters.


In [ ]:
def nested_season_selection(
    results: pd.DataFrame,
    metric: str = "exact_score_log_loss",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    seasons = sorted(results["season"].unique())
    selections: list[dict[str, float | int | str]] = []
    held_out_parts: list[pd.DataFrame] = []

    for evaluation_season in seasons:
        training = results[results["season"] < evaluation_season]
        evaluation = results[results["season"] == evaluation_season]

        if training.empty:
            continue

        training_aggregate = (
            training.groupby(
                [
                    "family",
                    "parameter_name",
                    "parameter_value",
                    "historical_weight",
                ],
                as_index=False,
            )[metric]
            .mean()
        )

        for family in training_aggregate["family"].unique():
            family_training = training_aggregate[
                training_aggregate["family"] == family
            ]
            best = family_training.loc[family_training[metric].idxmin()]

            selected_evaluation = evaluation[
                (evaluation["family"] == family)
                & np.isclose(
                    evaluation["parameter_value"],
                    best["parameter_value"],
                )
                & np.isclose(
                    evaluation["historical_weight"],
                    best["historical_weight"],
                )
            ].copy()

            selected_evaluation["selected_using_seasons_before"] = evaluation_season
            held_out_parts.append(selected_evaluation)

            selections.append(
                {
                    "evaluation_season": evaluation_season,
                    "family": family,
                    "parameter_name": best["parameter_name"],
                    "parameter_value": best["parameter_value"],
                    "historical_weight": best["historical_weight"],
                    f"training_{metric}": best[metric],
                }
            )

    held_out = (
        pd.concat(held_out_parts, ignore_index=True)
        if held_out_parts
        else pd.DataFrame()
    )
    return pd.DataFrame(selections), held_out


# Example:
# selections, nested_results = nested_season_selection(backtest_results)
# display(selections)
# display(
#     nested_results.groupby("family", as_index=False)[METRIC_COLUMNS].mean()
# )


## 11. Decision table

The final decision should be based on all of the following:

1. overall fixture-level log loss;
2. nested season holdout performance;
3. season-to-season stability;
4. paired bootstrap confidence intervals;
5. parameter stability;
6. computational cost and production complexity.

A candidate should not replace the existing blend merely because it wins by a tiny amount after tuning many parameter combinations.


In [ ]:
def model_decision_table(results: pd.DataFrame) -> pd.DataFrame:
    best = best_configuration_per_family(results).copy()

    best["complexity"] = best["family"].map(
        {
            "poisson": "low",
            "negative_binomial": "low-medium",
            "bivariate_poisson": "medium",
        }
    )
    best["interpretation"] = best["family"].map(
        {
            "poisson": "Independent equidispersed baseline",
            "negative_binomial": "Independent scores with overdispersion",
            "bivariate_poisson": "Positive score dependence via shared component",
        }
    )

    return best[
        [
            "family",
            "parameter_name",
            "parameter_value",
            "historical_weight",
            *METRIC_COLUMNS,
            "complexity",
            "interpretation",
        ]
    ]


# Example:
# display(model_decision_table(backtest_results))


## 12. Recommended execution order

1. Run the diagnostics.
2. Smoke-test 300–500 fixtures with `MAX_SCORE=60`.
3. Run the full grid and save fixture-level results to Parquet.
4. Inspect overall blend curves.
5. Inspect season-level stability.
6. Run nested season selection.
7. Bootstrap the best Negative Binomial and Bivariate Poisson configurations against the current Poisson blend.
8. Only promote a candidate when it improves held-out performance consistently and materially.

### Important modelling note

The classical bivariate Poisson permits only non-negative covariance. If the residual relationship is negative or more complex than a shared game-state component, a different dependence model would eventually be required. The empirical historical matrix may already capture much of this dependence, so a small or zero optimal shared fraction would still be an informative result.
